
### other issues that need addressing:
- review docx and txt renderers and identify where markdown is being produced - and enclose any  underscore containing items into backticks. so basically field names etc. otherwise in nicegui they are incorrectly rendered as markdown.
- in main nicegui table and in researcher card, somehow the line spacing is too large. in the attempt history table it's perfect.
- in main nicegui table the row clicked on doesn't get highlighted which is confusing.
- when nicegui table row is selected/unselected multiple times, this expands and collapses attempts table which is not intuitive. what should happen is that once it's selected, the attempt history should be expanded, and any future clicks will be idempotent.
when clicking on a different row, the selection will change.
therefore when for the first time a row has been selected, rows never get unselected
and therefore attempt history persists.
- in the search box when i remove the value, the search doesn't get reset. **already addressed in commit: `eeeaeacd8aef6d425c27935d2b00cd8777c196fa`. only remains to wire in spec.**
every case above must have a dedicated roundtrip test.


In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

CARDS_TEST_PATH = REPOSITORY_ROOT / "tests" / "test_cards.py"
CONTROL_CENTRE_TEST_PATH = (
    REPOSITORY_ROOT
    / "src"
    / "detours"
    / "detour_ai_augment"
    / "tests"
    / "test_control_centre.py"
)
CONTROL_CENTRE_E2E_TEST_PATH = CONTROL_CENTRE_TEST_PATH.with_name(
    "test_control_centre_e2e.py"
)
PYTEST_NODEIDS = {
    "literal_labels_outputs": (
        f"{CARDS_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_underscore_field_labels_round_trip_in_txt_and_docx"
    ),
    "literal_labels_browser": (
        f"{CONTROL_CENTRE_E2E_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_underscore_field_labels_render_literally_in_researcher_card"
    ),
    "compact_spacing": (
        f"{CONTROL_CENTRE_E2E_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_main_grid_and_researcher_card_use_compact_line_spacing"
    ),
    "row_highlight": (
        f"{CONTROL_CENTRE_E2E_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_selected_researcher_row_is_highlighted"
    ),
    "persistent_history": (
        f"{CONTROL_CENTRE_E2E_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_researcher_selection_and_attempt_history_are_idempotent"
    ),
    "clear_search": (
        f"{CONTROL_CENTRE_TEST_PATH.relative_to(REPOSITORY_ROOT)}"
        "::test_cleared_search_stays_empty_during_timer_refresh"
    ),
}
PASSED_PYTEST_NODEIDS: set[str] = set()

def run_pytest(*nodeids: str) -> None:
    pending = [
        nodeid for nodeid in nodeids if nodeid not in PASSED_PYTEST_NODEIDS
    ]
    if not pending:
        return
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", *pending],
        cwd=REPOSITORY_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=900,
        check=False,
    )
    assert completed.returncode == 0, completed.stdout
    PASSED_PYTEST_NODEIDS.update(pending)

run_pytest(
    PYTEST_NODEIDS["literal_labels_outputs"],
    PYTEST_NODEIDS["literal_labels_browser"],
)


In [ ]:
run_pytest(PYTEST_NODEIDS["compact_spacing"])


In [ ]:
run_pytest(PYTEST_NODEIDS["row_highlight"])


In [ ]:
run_pytest(PYTEST_NODEIDS["persistent_history"])


In [ ]:
run_pytest(PYTEST_NODEIDS["clear_search"])
